You are given infrastructure usage records.

Each record has:
- service
- cpu utilization
- memory utilization

Write Python code to calculate average CPU and memory by service.

In [2]:
import pandas as pd
records = [
    {"service": "checkout", "cpu": 72, "memory": 68},
    {"service": "checkout", "cpu": 81, "memory": 74},
    {"service": "search", "cpu": 45, "memory": 52},
    {"service": "search", "cpu": 91, "memory": 88},
]

df = pd.DataFrame(records)
result = (
    df.groupby("service", as_index= False)
     .agg(
         avg_cpu = ("cpu", "mean"),
         avg_memory=("memory", "mean"),
     )
    )

result["avg_cpu"] = result["avg_cpu"].round(2)
result["avg_memory"] = result["avg_memory"].round(2)

print(result)

    service  avg_cpu  avg_memory
0  checkout     76.5        71.0
1    search     68.0        70.0


In [5]:
records = [
    {"service": "checkout", "cpu": 72, "memory": 68},
    {"service": "checkout", "cpu": 81, "memory": 74},
    {"service": "search", "cpu": 45, "memory": 52},
    {"service": "search", "cpu": 91, "memory": 88},
]

grouped = {}

for row in records:
    service = row["service"]
    
    # setdefault checks if 'service' exists. 
    # If not, it creates it with the dict we provided.
    # Then it returns that dict so we can append to it immediately.
    m = grouped.setdefault(service, {"cpu": [], "memory": []})
    
    m["cpu"].append(row["cpu"])
    m["memory"].append(row["memory"])

for service, metrics in grouped.items():
    avg_cpu = sum(metrics["cpu"]) / len(metrics["cpu"])
    avg_memory = sum(metrics["memory"]) / len(metrics["memory"])

    print({
        "service": service,
        "avg_cpu": round(avg_cpu, 2),
        "avg_memory": round(avg_memory, 2),
    })

{'service': 'checkout', 'avg_cpu': 76.5, 'avg_memory': 71.0}
{'service': 'search', 'avg_cpu': 68.0, 'avg_memory': 70.0}


Now identify services that may be capacity risks.

A service is risky if:
- average CPU is above 75
OR
- average memory is above 75

In [7]:
from collections import defaultdict

records = [
    {"service": "checkout", "cpu": 72, "memory": 68},
    {"service": "checkout", "cpu": 81, "memory": 74},
    {"service": "search", "cpu": 45, "memory": 52},
    {"service": "search", "cpu": 91, "memory": 88},
    {"service": "billing", "cpu": 30, "memory": 35},
]

grouped = defaultdict(lambda: {"cpu": [], "memory": []})

for row in records:
    service = row["service"]
    grouped[service]["cpu"].append(row["cpu"])
    grouped[service]["memory"].append(row["memory"])

risk_list = []

for service, metrics in grouped.items():
    avg_cpu = sum(metrics["cpu"]) / len(metrics["cpu"])
    avg_memory = sum(metrics["memory"]) / len(metrics["memory"])

    if avg_cpu > 75 or avg_memory > 75:
        risk_list.append({
            "service": service,
            "avg_cpu": round(avg_cpu, 2),
            "avg_memory": round(avg_memory, 2),
            "reason": "High average utilization",
        })

print(risk_list)

[{'service': 'checkout', 'avg_cpu': 76.5, 'avg_memory': 71.0, 'reason': 'High average utilization'}]


Write a function to calculate P95 utilization.

In [11]:
def percentile(values, percentile_value):
    if not values:
        return None

    sorted_values = sorted(values)
    index = round((percentile_value / 100) * (len(sorted_values) - 1))
    
    return sorted_values[index]


cpu_values = [45, 52, 60, 75, 80, 92, 95, 97, 99]

p95_cpu = percentile(cpu_values, 95)

print(p95_cpu)

99


In [14]:
import pandas as pd

records = [
    {"service": "checkout", "cpu": 72, "memory": 68},
    {"service": "checkout", "cpu": 81, "memory": 74},
    {"service": "checkout", "cpu": 92, "memory": 85},
    {"service": "search", "cpu": 45, "memory": 52},
    {"service": "search", "cpu": 91, "memory": 88},
    {"service": "search", "cpu": 78, "memory": 76},
]

df = pd.DataFrame(records)

result = (
    df.groupby("service", as_index=False)
      .agg(
          avg_cpu=("cpu", "mean"),
          p95_cpu=("cpu", lambda x: x.quantile(0.95)),
          avg_memory=("memory", "mean"),
          p95_memory=("memory", lambda x: x.quantile(0.95)),
      )
)

result = result.round(2)

print(result)

    service  avg_cpu  p95_cpu  avg_memory  p95_memory
0  checkout    81.67     90.9       75.67        83.9
1    search    71.33     89.7       72.00        86.8


Given current P95 utilization and a capacity threshold,
calculate available headroom.

In [12]:
services = [
    {"service": "checkout", "p95_cpu": 82, "threshold": 90},
    {"service": "search", "p95_cpu": 65, "threshold": 90},
    {"service": "billing", "p95_cpu": 91, "threshold": 90},
]

for row in services:
    headroom = row["threshold"] - row["p95_cpu"]

    status = "ok"
    if headroom <= 0:
        status = "over threshold"
    elif headroom <= 10:
        status = "watch"

    print({
        "service": row["service"],
        "p95_cpu": row["p95_cpu"],
        "headroom": headroom,
        "status": status,
    })

{'service': 'checkout', 'p95_cpu': 82, 'headroom': 8, 'status': 'watch'}
{'service': 'search', 'p95_cpu': 65, 'headroom': 25, 'status': 'ok'}
{'service': 'billing', 'p95_cpu': 91, 'headroom': -1, 'status': 'over threshold'}


Given forecasted and actual usage by month,
calculate variance percentage.

In [13]:
rows = [
    {"month": "2026-01", "forecast": 1000, "actual": 1080},
    {"month": "2026-02", "forecast": 1200, "actual": 1140},
    {"month": "2026-03", "forecast": 1300, "actual": 1500},
]

for row in rows:
    forecast = row["forecast"]
    actual = row["actual"]

    variance = actual - forecast
    variance_pct = (variance / forecast) * 100 if forecast else None

    print({
        "month": row["month"],
        "forecast": forecast,
        "actual": actual,
        "variance": variance,
        "variance_pct": round(variance_pct, 2),
    })

{'month': '2026-01', 'forecast': 1000, 'actual': 1080, 'variance': 80, 'variance_pct': 8.0}
{'month': '2026-02', 'forecast': 1200, 'actual': 1140, 'variance': -60, 'variance_pct': -5.0}
{'month': '2026-03', 'forecast': 1300, 'actual': 1500, 'variance': 200, 'variance_pct': 15.38}


Forecast versus actual is the feedback loop.

If actual is much higher than forecast, I want to know whether the driver
was product growth, a launch, seasonality, bad assumptions, missing telemetry,
or workload behavior.

The goal is not to defend the model. The goal is to improve the planning
process.

# Pandas

In [15]:
import pandas as pd
from io import StringIO


csv_data = """
date,service,environment,cpu,memory,forecast_cpu,allocated_cpu,cost
2026-01-01,checkout,prod,72,68,70,100,120.50
2026-01-02,checkout,prod,81,74,75,100,125.00
2026-01-03,checkout,prod,92,85,80,100,132.25
2026-01-01,search,prod,45,52,50,100,90.00
2026-01-02,search,prod,91,88,65,100,118.75
2026-01-03,search,prod,78,76,70,100,110.10
2026-01-01,billing,prod,30,35,35,100,80.00
2026-01-02,billing,prod,,38,40,100,82.50
2026-01-03,billing,prod,28,bad_data,38,100,79.25
"""


# 1. Read telemetry CSV
df = pd.read_csv(StringIO(csv_data))


# 2. Clean column names
df.columns = [col.strip().lower() for col in df.columns]


# 3. Convert dates
df["date"] = pd.to_datetime(df["date"], errors="coerce")


# 4. Convert numeric columns safely
numeric_columns = [
    "cpu",
    "memory",
    "forecast_cpu",
    "allocated_cpu",
    "cost",
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")


# 5. Fill missing CPU/memory with service-level average
for col in ["cpu", "memory"]:
    df[col] = (
        df.groupby("service")[col]
          .transform(lambda x: x.fillna(x.mean()))
    )


# 6. Drop records still missing critical fields
df = df.dropna(
    subset=[
        "date",
        "service",
        "cpu",
        "memory",
        "forecast_cpu",
        "allocated_cpu",
    ]
)


# 7. Calculate row-level capacity features
df["cpu_headroom"] = df["allocated_cpu"] - df["cpu"]
df["forecast_variance"] = df["cpu"] - df["forecast_cpu"]

df["forecast_variance_pct"] = (
    df["forecast_variance"] / df["forecast_cpu"] * 100
)


# 8. Aggregate by service
summary = (
    df.groupby("service", as_index=False)
      .agg(
          avg_cpu=("cpu", "mean"),
          p95_cpu=("cpu", lambda x: x.quantile(0.95)),
          avg_memory=("memory", "mean"),
          p95_memory=("memory", lambda x: x.quantile(0.95)),
          avg_headroom=("cpu_headroom", "mean"),
          avg_forecast_variance_pct=("forecast_variance_pct", "mean"),
          total_cost=("cost", "sum"),
      )
)


# 9. Round output for reporting
summary = summary.round(2)


# 10. Add capacity risk logic
def classify_capacity_risk(row):
    if row["p95_cpu"] >= 90:
        return "high_capacity_risk"

    if row["avg_headroom"] <= 15:
        return "watch_headroom"

    if row["avg_cpu"] < 35:
        return "rightsizing_candidate"

    return "normal"


summary["capacity_status"] = summary.apply(
    classify_capacity_risk,
    axis=1,
)


# 11. Add business recommendation
def recommendation(row):
    if row["capacity_status"] == "high_capacity_risk":
        return "Review scaling plan and forecast next quarter demand"

    if row["capacity_status"] == "watch_headroom":
        return "Monitor closely and validate upcoming demand"

    if row["capacity_status"] == "rightsizing_candidate":
        return "Review for underutilization and possible rightsizing"

    return "No immediate action"


summary["recommendation"] = summary.apply(
    recommendation,
    axis=1,
)


# 12. Print clean report
print("\n=== Cleaned Telemetry ===")
print(df)

print("\n=== Capacity Summary By Service ===")
print(summary)


=== Cleaned Telemetry ===
        date   service environment   cpu  memory  forecast_cpu  allocated_cpu  \
0 2026-01-01  checkout        prod  72.0    68.0            70            100   
1 2026-01-02  checkout        prod  81.0    74.0            75            100   
2 2026-01-03  checkout        prod  92.0    85.0            80            100   
3 2026-01-01    search        prod  45.0    52.0            50            100   
4 2026-01-02    search        prod  91.0    88.0            65            100   
5 2026-01-03    search        prod  78.0    76.0            70            100   
6 2026-01-01   billing        prod  30.0    35.0            35            100   
7 2026-01-02   billing        prod  29.0    38.0            40            100   
8 2026-01-03   billing        prod  28.0    36.5            38            100   

     cost  cpu_headroom  forecast_variance  forecast_variance_pct  
0  120.50          28.0                2.0               2.857143  
1  125.00          19.0   